# 🚀 Lesson 15: Capstone Final — Dashboard, Memory & Deploy

**This is it!** We complete the Interview Copilot with a beautiful dashboard, AI memory, and production deployment.

By the end of this lesson, you'll have a portfolio-ready project to show any interviewer!

In [ ]:
import os, json, glob
from pathlib import Path
from dotenv import load_dotenv
from rich.console import Console
load_dotenv()
console = Console()
Path('./copilot_data').mkdir(exist_ok=True)
console.print('[bold green]🏁 Final Lesson — Interview Copilot Complete Build[/bold green]')

## 🧠 Part 1: Long-Term Memory with ChromaDB

In [ ]:
# !pip install chromadb sentence-transformers
import chromadb
from chromadb.utils import embedding_functions

# Setup ChromaDB with sentence transformers
chroma_client = chromadb.PersistentClient(path='./copilot_data/chroma_db')
embedding_fn = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name='all-MiniLM-L6-v2'
)

# Collections
sessions_collection = chroma_client.get_or_create_collection(
    name='interview_sessions',
    embedding_function=embedding_fn
)

def store_session_in_memory(session_data: dict):
    """Store session results in ChromaDB for long-term memory."""
    doc_text = f"""
    User: {session_data['user']}
    Problem: {session_data['problem']}
    Score: {session_data['final_score']}
    Hints Used: {session_data['hints_used']}
    Duration: {session_data['duration_seconds']}s
    """
    sessions_collection.add(
        documents=[doc_text],
        metadatas=[session_data],
        ids=[session_data['session_id']]
    )
    console.print(f"[green]✅ Session {session_data['session_id']} stored in memory[/green]")


def get_weak_topics(user_name: str, top_n: int = 3) -> list:
    """Query memory to find topics the user struggles with."""
    results = sessions_collection.query(
        query_texts=[f'{user_name} struggled'],
        n_results=min(10, sessions_collection.count()),
        where={'user': user_name} if sessions_collection.count() > 0 else None
    )
    
    topic_scores = {}
    if results['metadatas']:
        for meta in results['metadatas'][0]:
            topic = meta.get('problem', 'Unknown')
            score = meta.get('final_score', 0)
            if topic not in topic_scores:
                topic_scores[topic] = []
            topic_scores[topic].append(score)
    
    # Find topics with lowest avg score
    avg_scores = {t: sum(s)/len(s) for t, s in topic_scores.items()}
    weak = sorted(avg_scores.items(), key=lambda x: x[1])[:top_n]
    return [t for t, _ in weak]

# Demo: store a sample session
sample_session = {
    'session_id': 'demo001',
    'user': 'Avnish',
    'problem': 'Two Sum',
    'final_score': 0.85,
    'hints_used': 1,
    'duration_seconds': 420,
    'attempts': []
}
store_session_in_memory(sample_session)
console.print('[cyan]Weak topics for Avnish:[/cyan]', get_weak_topics('Avnish'))

## 📊 Part 2: Performance Dashboard (Streamlit)

In [ ]:
# This creates a dashboard.py file you run separately: streamlit run dashboard.py
dashboard_code = '''
import streamlit as st
import json, glob
import plotly.graph_objects as go
import plotly.express as px
import pandas as pd
from pathlib import Path

st.set_page_config(page_title="Interview Copilot", page_icon="🤖", layout="wide")

# Custom CSS
st.markdown("""
<style>
.main-header {font-size: 2.5rem; font-weight: bold; color: #00ff88; text-align: center;}
.metric-card {background: #1e1e2e; padding: 20px; border-radius: 12px; border-left: 4px solid #00ff88;}
</style>
""", unsafe_allow_html=True)

st.markdown('<div class="main-header">🤖 Interview Copilot Dashboard</div>', unsafe_allow_html=True)

# Load session data
sessions = []
for f in glob.glob('./copilot_data/session_*.json'):
    with open(f) as fp:
        sessions.append(json.load(fp))

if not sessions:
    st.warning("No sessions yet! Complete a practice session first.")
    st.stop()

df = pd.DataFrame(sessions)

# --- Metrics Row ---
col1, col2, col3, col4 = st.columns(4)
with col1:
    st.metric("Total Sessions", len(sessions))
with col2:
    avg_score = df["final_score"].mean() * 100
    st.metric("Avg Score", f"{avg_score:.1f}%")
with col3:
    problems_solved = df[df["final_score"] >= 0.7]["problem"].nunique()
    st.metric("Problems Solved (≥70%)", problems_solved)
with col4:
    avg_hints = df["hints_used"].mean()
    st.metric("Avg Hints/Session", f"{avg_hints:.1f}")

st.divider()

# --- Score Over Time ---
col1, col2 = st.columns(2)
with col1:
    st.subheader("📈 Score Trend")
    fig = px.line(df, y="final_score", title="Score Over Sessions",
                  color_discrete_sequence=["#00ff88"])
    fig.update_yaxes(range=[0, 1], tickformat=".0%")
    st.plotly_chart(fig, use_container_width=True)

with col2:
    st.subheader("📚 Performance by Problem")
    problem_avg = df.groupby("problem")["final_score"].mean().reset_index()
    fig2 = px.bar(problem_avg, x="problem", y="final_score",
                  color="final_score", color_continuous_scale="RdYlGn")
    fig2.update_yaxes(range=[0, 1], tickformat=".0%")
    st.plotly_chart(fig2, use_container_width=True)

st.divider()

# --- Recent Sessions Table ---
st.subheader("📋 Session History")
display_df = df[["session_id", "problem", "final_score", "hints_used", "duration_seconds"]].copy()
display_df["final_score"] = (display_df["final_score"] * 100).round(1).astype(str) + "%"
display_df["duration_seconds"] = display_df["duration_seconds"].astype(str) + "s"
st.dataframe(display_df, use_container_width=True)
'''

with open('./lesson-15-capstone-final/dashboard.py', 'w') as f:
    f.write(dashboard_code)

console.print('[green]✅ dashboard.py created![/green]')
console.print('[yellow]▶️  Run: streamlit run lesson-15-capstone-final/dashboard.py[/yellow]')

## 🐳 Part 3: Docker + FastAPI Production Setup

In [ ]:
# Create FastAPI app
fastapi_code = '''
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from typing import Optional
import json, os
from pathlib import Path
from dotenv import load_dotenv
load_dotenv()

app = FastAPI(title="Interview Copilot API", version="1.0.0")

class ChatRequest(BaseModel):
    user_name: str
    problem_id: str
    message: str
    session_id: Optional[str] = None

class CodeSubmitRequest(BaseModel):
    session_id: str
    problem_id: str
    code: str

@app.get("/")
def root():
    return {"message": "🤖 Interview Copilot API Running!", "status": "ready"}

@app.get("/problems")
def list_problems():
    return {"problems": ["Two Sum", "Valid Parentheses", "Binary Search"]}

@app.post("/chat")
async def chat(req: ChatRequest):
    # In full version: runs AutoGen agent
    return {
        "session_id": req.session_id or "new_session",
        "reply": f"Great approach on {req.problem_id}! Keep going.",
        "hints_used": 0
    }

@app.post("/submit")
async def submit_code(req: CodeSubmitRequest):
    # In full version: runs sandbox
    return {"passed": 3, "total": 3, "score": 1.0, "feedback": "Perfect solution!"}

@app.get("/dashboard/{user_name}")
def user_dashboard(user_name: str):
    sessions = []
    for f in Path("./copilot_data").glob("session_*.json"):
        with open(f) as fp:
            s = json.load(fp)
            if s.get("user") == user_name:
                sessions.append(s)
    avg_score = sum(s["final_score"] for s in sessions) / len(sessions) if sessions else 0
    return {"user": user_name, "sessions": len(sessions), "avg_score": round(avg_score, 2)}
'''

Path('./lesson-15-capstone-final').mkdir(exist_ok=True)
with open('./lesson-15-capstone-final/api.py', 'w') as f:
    f.write(fastapi_code)

# Dockerfile
dockerfile = '''
FROM python:3.11-slim

WORKDIR /app
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

COPY . .
RUN mkdir -p copilot_data

EXPOSE 8000
CMD ["uvicorn", "lesson-15-capstone-final.api:app", "--host", "0.0.0.0", "--port", "8000"]
'''

with open('./Dockerfile', 'w') as f:
    f.write(dockerfile)

console.print('[green]✅ FastAPI + Dockerfile created![/green]')
console.print('[yellow]▶️  Run locally: uvicorn lesson-15-capstone-final.api:app --reload[/yellow]')
console.print('[yellow]🐳 Docker: docker build -t interview-copilot . && docker run -p 8000:8000 interview-copilot[/yellow]')

## 🎉 CONGRATULATIONS!

You have completed the AI Agents Course and built a production-ready Interview Copilot!

### What You Built:
- ✅ 15-lesson progressive curriculum
- ✅ Microsoft AutoGen multi-agent system
- ✅ Azure OpenAI + Whisper integration
- ✅ Live code execution sandbox
- ✅ ChromaDB long-term memory
- ✅ Streamlit dashboard
- ✅ FastAPI production backend
- ✅ Docker containerization

### 🚀 Next Steps:
1. Push to GitHub with a great README
2. Deploy to Azure Container Apps
3. Add to your portfolio at avnish1505.github.io
4. Write a LinkedIn post about it!

**You're ready for AI Engineer internship interviews, bhai! 🔥**